# 29. The weekly refresh: a dated forecast to grade later

**Run this notebook once a week.** It:

1. downloads only the results and boxscores that are new since the last run;
2. checks them (every game has a boxscore, goals match, roster changes you may need to log);
3. reruns notebook 28's in-season update with its **frozen settings** (`data/forecast_log/settings.json`);
4. runs the **reality scenario**: from today, with every logged trade through yesterday and the players in `data/sit_out_2026-27.csv` sitting;
5. saves one **dated entry** in `data/forecast_log/<yesterday's date>/`. It's **never overwritten**: a second run on the same day refuses to save.

**Why the entries matter:** a later notebook will grade them against what really happens. So each entry holds everything needed to understand and rebuild it: the forecasts, the settings, the evidence and a fingerprint of the code.

**What it never does:**

- change notebooks 01-28, the app, the simulator bundle or `trades_2026-27.csv`;
- download MoneyPuck files: you do that by hand, and this notebook only checks their dates.

**The model** is notebook 28's, not new: `src/refresh.py` only calls `inseason.py`, `season.py` and `scoring.py`.

## 1. Setup: dates, frozen settings, your files

In [1]:
import sys
sys.path.append("../src")

import os
import json
import time
from datetime import date, datetime, timedelta
from pathlib import Path
import numpy as np
import pandas as pd

import edge_data as E
import simulate as S
import season as SE
import refresh as R

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 220)

DATA = Path("../data")
LOG = DATA / "forecast_log"                              # the forecast log: one folder per evidence date
BOX_DIR = DATA / "schedule_raw" / "2026-27" / "boxscores"
DAY_DIR = DATA / "schedule_raw" / "2026-27" / "days"
SIT_PATH = DATA / "sit_out_2026-27.csv"                  # yours: who sits in the reality scenario
ACK_PATH = DATA / "drift_ack_2026-27.csv"                # yours: roster changes you've looked at and accept
MONEYPUCK = [DATA / "skaters26-27.csv", DATA / "goalies26-27.csv"]   # you download these by hand
OPENING_NIGHT = SE.OPENING_NIGHT
OLD_LOG = LOG / "updated_projections_2026-11-10.csv"     # notebook 28's file, converted once into the first entry

# Today is the computer's date. Evidence runs through yesterday; the forecast starts today.
TODAY = os.environ.get("HOCKEY_TODAY") or date.today().isoformat()     # HOCKEY_TODAY: only for testing
EVIDENCE = (date.fromisoformat(TODAY) - timedelta(days=1)).isoformat()
START = TODAY
RUN_STARTED = datetime.now().isoformat(timespec="seconds")

ST = R.load_settings(LOG / "settings.json")              # stops here with a clear message if it's missing
B = SE.load_bundle(DATA / "simulator_2026-27", DATA / "trades_2026-27.csv")
INPUTS = S.draw_inputs(pd.read_csv(DATA / "dev_residuals.csv"), pd.read_csv(DATA / "dev_ranges.csv"), center_pools=True)
FIT, POOLS = INPUTS["fit"], B["pools"]

SIT = R.ensure_csv(SIT_PATH, ["playerId", "name", "reason", "since"],
                   [(8477946, "Dylan Larkin", "suspended / holdout", "2026-11"), (8482762, "Simon Edvinsson", "unsigned RFA", "2026-09-29")])
ACK = R.ensure_csv(ACK_PATH, ["playerId", "name", "note"])

ALREADY_SAVED = R.entry_exists(LOG, EVIDENCE)
print(f"Today {TODAY}: evidence through {EVIDENCE}, forecast from {START}.")
print("Frozen settings (notebook 28): " + ", ".join(f"{k} {ST[k]}" for k in ["version", "skaters", "goalies", "teams", "g0", "beta", "n_sims", "scenario_seed"]))
print(f"Sitting out: {', '.join(SIT['name'])}. Roster changes acknowledged: {len(ACK)}.")
if ALREADY_SAVED:
    print(f"\nAn entry for {EVIDENCE} already exists: this run will NOT save. Everything else still runs, so you can look.")

Created ../data/sit_out_2026-27.csv (2 rows). It's yours to edit; this notebook never writes to it again.
Created ../data/drift_ack_2026-27.csv (0 rows). It's yours to edit; this notebook never writes to it again.
Today 2026-11-11: evidence through 2026-11-10, forecast from 2026-11-11.
Frozen settings (notebook 28): version whole, skaters True, goalies False, teams True, g0 49.20549472603196, beta 0.37905941334543736, n_sims 10000, scenario_seed 28
Sitting out: Dylan Larkin, Simon Edvinsson. Roster changes acknowledged: 0.


## 2. New results and boxscores (only what's new)

Two independent sources:

- **The daily scores** (`api-web.nhle.com/v1/score/<date>`, one call per day). A day is saved only once every game on it is final, so a day with a game still going is retried next time.
- **The boxscores** (one call per game). Only games not saved yet are downloaded.

Both use the same polite pace as before (0.3 seconds between calls). Both folders are gitignored, and finished games never change.

In [2]:
not_saved = E.download_score_days(R.day_range(OPENING_NIGHT, EVIDENCE), DAY_DIR)
results = E.score_day_results(DAY_DIR)
final_ids = results.loc[results["final"] & (results["date"] <= EVIDENCE), "game_id"].tolist()
failed = E.download_boxscores(final_ids, BOX_DIR)
box = E.boxscore_lines(BOX_DIR)
box = box[box["game_id"].isin(final_ids)]
print(f"Results through {EVIDENCE}: {len(final_ids)} games final; boxscores saved for {box['game_id'].nunique()}")

Score days: 43 already saved, 0 downloaded
Boxscores: 292 already saved, 0 downloaded
Results through 2026-11-10: 292 games final; boxscores saved for 292


## 3. Checks before anything runs

1. **Every day through yesterday is complete**, and **every final game has a boxscore.** If not, the notebook stops here: run it again later.
2. **Team goals in the boxscores = the daily scores' results**, game by game. A shootout winner's extra "goal" counts in the standings but isn't a player's goal, so it's taken off.
3. **MoneyPuck files** (you download them by hand): their dates, with a warning if they're more than 7 days older than the newest boxscore. Only the "with skater WAR" version uses them.
4. **Roster drift:** players who played for a different team than their roster team after the logged trades.
   - **(a) A different NHL team** (a trade or waiver claim not logged yet) **blocks the save** until you either:
     - add it to `trades_2026-27.csv`, or
     - list his playerId in `data/drift_ack_2026-27.csv`.

     **A waiver claim** goes in the trades file as a one-player row: a new id like `W1`, date = his first game with the new team, note "waiver claim". The simulator moves him exactly like a traded player.
   - **(b) On no roster at all** (call-ups): a warning only.

In [3]:
problems = []
if not_saved:
    problems.append(f"days not complete yet (a game wasn't final): {not_saved}")
missing, postponed = R.missing_boxscores(results, box, EVIDENCE)
if failed or len(missing):
    problems.append(f"final games without a boxscore: {sorted(set(failed) | set(missing['game_id']))}")
if problems:
    raise RuntimeError("Stopping before anything runs: " + "; ".join(problems) + ". Run the notebook again later.")
print(f"1. Every day through {EVIDENCE} is complete and every final game has a boxscore. "
      f"Postponed so far: {len(postponed)}" + (f" ({', '.join(map(str, postponed['game_id']))})" if len(postponed) else ""))

GOALS = R.goal_check(results, box)
print(f"2. Team goals, boxscores vs. results: {2 * len(final_ids) - len(GOALS)} of {2 * len(final_ids)} team-games match"
      + ("" if GOALS.empty else "; mismatches:\n" + GOALS.to_string(index=False)))

MP = R.file_dates(MONEYPUCK, box["date"].max())
print("3. MoneyPuck files (their date = when you downloaded them):")
print(MP.to_string(index=False))
if MP["stale"].any():
    print("   WARNING: refresh the MoneyPuck files by hand. This doesn't stop the run; they only feed the skater WAR version.")

rosters = pd.read_csv(DATA / "rosters_2026-27.csv")
DRIFT, CALLUPS, ACKED = R.roster_drift(box, rosters, B["trades"], EVIDENCE, ACK["playerId"].tolist())
print(f"4a. Playing for a different NHL team than the roster + logged trades say: {len(DRIFT)}"
      + ("" if DRIFT.empty else " (THE SAVE IS BLOCKED until each one is logged or acknowledged)\n" + DRIFT.to_string(index=False)))
if len(ACKED):
    print(f"    Acknowledged in {ACK_PATH.name}: {', '.join(ACKED['name'])}")
print(f"4b. On no NHL roster (warning only): {len(CALLUPS)}" + ("" if CALLUPS.empty else "\n" + CALLUPS.to_string(index=False)))

1. Every day through 2026-11-10 is complete and every final game has a boxscore. Postponed so far: 0
2. Team goals, boxscores vs. results: 583 of 584 team-games match; mismatches:
   game_id       date team  boxscore  result
2026020010 2026-10-01  NYR         4       5
3. MoneyPuck files (their date = when you downloaded them):
            file       date  stale note
skaters26-27.csv 2026-11-10  False     
goalies26-27.csv 2026-11-10  False     
4a. Playing for a different NHL team than the roster + logged trades say: 0
4b. On no NHL roster (warning only): 4
 playerId          name team_now roster_team first_game_with_team_now
  8482121   D. Peterson      STL         NaN               2026-10-13
  8484900      C. Black      FLA         NaN               2026-10-22
  8485468     T. Taylor      DAL         NaN               2026-10-29
  8486130 D. Hryckowian      DAL         NaN               2026-10-25


## 4. The first entry: notebook 28's Nov 9 forecast (only once)

Notebook 28 saved one file (`updated_projections_2026-11-10.csv`, evidence through Nov 9). Here it becomes the first entry, `data/forecast_log/2026-11-09/`, **with its numbers unchanged**.

- **What notebook 28 didn't save:** rest-of-season ranges, full-season points and every team column. Those are recomputed from the Nov 9 evidence with the frozen settings.
- **The proof first:** the recomputed rates, ESS and team bonuses must equal notebook 28's to 1e-9, otherwise only the copied columns are kept.
- **Detroit's line** is compared with notebook 28's printed 80.8 points and 10.2%, and reported.
- **The metadata says** `"backfilled": true`, when it was created, and which columns were copied vs. recomputed.

This cell does nothing once the entry exists.

In [4]:
def run_week(through, start):
    """The whole update and reality scenario for one evidence date (the same steps as sections 5-6)."""
    Bn = R.with_evidence(B, results, box, through, start)
    trades_in = sorted(B["trades"].loc[B["trades"]["date"] <= through, "trade_id"].unique())
    sit = SIT["playerId"].astype(int).tolist()
    lu = R.reality_lineup(Bn, trades_in, sit)
    P, left = R.player_table(Bn, lu, start, MONEYPUCK[0], MONEYPUCK[1], ST["goalie_repl_gap"])
    U_def, Z_def = R.player_update(P, ST, FIT, POOLS, with_skater_war=False)
    U_war, Z_war = R.player_update(P, ST, FIT, POOLS, with_skater_war=True)
    bonus, runs = R.team_bonus_update(Bn, B, start, trades_in, sit, P, Z_def, left, ST)
    common = dict(start_date=start, trade_ids=trades_in, sit_out=sit, n_sims=ST["n_sims"], seed=ST["scenario_seed"], stat_lines=True)
    ids = P["playerId"].to_numpy()
    upd = SE.run_scenario(Bn, **common, updated={"playerId": ids, "z": Z_def, "bonus": bonus.to_dict()})
    uwar = SE.run_scenario(Bn, **common, updated={"playerId": ids, "z": Z_war, "bonus": bonus.to_dict()})
    return {"B": Bn, "trades": trades_in, "sit": sit, "lineup": lu, "P": P, "U_def": U_def, "U_war": U_war, "Z_def": Z_def,
            "bonus": bonus, "runs": runs, "upd": upd, "uwar": uwar}

BACKFILL = "2026-11-09"
if R.entry_exists(LOG, BACKFILL) or not OLD_LOG.exists():
    print(f"The {BACKFILL} entry already exists (or there's no notebook 28 file): nothing to do.")
else:
    W9 = run_week(BACKFILL, "2026-11-10")
    old = pd.read_csv(OLD_LOG)
    o = old.set_index("playerId")
    u, w = W9["U_def"].set_index("playerId"), W9["U_war"].set_index("playerId")
    strict = {"upd_war60": u["war60"], "upd_g60": u["g60"], "upd_a60": u["a60"], "upd_ess": u["ess"]}
    war_cols = {"upd_war_war60": w["war60"], "upd_war_g60": w["g60"], "upd_war_a60": w["a60"], "upd_war_ess": w["ess"]}
    gap = lambda cols: max(float(np.nanmax(np.abs(o[c] - v.reindex(o.index)))) for c, v in cols.items())
    b_old = o.groupby("team_now")["team_bonus_upd"].first()
    gap_bonus = float(np.abs(b_old - W9["bonus"].reindex(b_old.index)).max())
    gap_def, gap_war = gap(strict), gap(war_cols)
    faithful = gap_def < 1e-9 and gap_bonus < 1e-9
    print(f"Recomputed vs. notebook 28's file: default version largest gap {gap_def:.1e}, team bonuses {gap_bonus:.1e}, "
          f"skater WAR version {gap_war:.1e} (the rule: under 1e-9)")
    det = W9["upd"]["table"].set_index("team").loc["DET"]
    print(f"Detroit, recomputed: {det['points']:.1f} points, {det['playoff_pct']:.1f}% playoffs (notebook 28 printed 80.8, 10.2%)")
    copied = list(old.columns)
    players = old.copy()
    recomputed = []
    if faithful:
        lines_def, _ = SE.season_lines(W9["upd"], W9["B"])
        lines_war, _ = SE.season_lines(W9["uwar"], W9["B"])
        players = R.add_rest_lines(players, lines_def, lines_war)
        if gap_war >= 1e-9:
            players["rest_p_war_version"] = np.nan          # the MoneyPuck file changed since notebook 28
        recomputed = [c for c in R.REST_COLS if players[c].notna().any()]
        teams = R.team_entry(W9["upd"], W9["uwar"], W9["runs"]["bonus0"], W9["bonus"])
    else:
        teams = pd.DataFrame({"team": b_old.index, "team_bonus_upd": b_old.to_numpy()})
    meta = {"evidence_through": BACKFILL, "start": "2026-11-10", "backfilled": True, "created": TODAY,
            "source": f"{OLD_LOG.name} (notebook 28, commit 1510414)",
            "columns_copied": copied, "columns_recomputed": recomputed + ([c for c in teams.columns if c != "team"] if faithful else []),
            "recomputed_on": TODAY, "faithful_to_notebook_28": faithful,
            "largest_gaps": {"default": gap_def, "team_bonus": gap_bonus, "skater_war_version": gap_war},
            "detroit_recomputed": {"points": float(det["points"]), "playoff_pct": float(det["playoff_pct"])},
            "settings": ST, "seed": ST["scenario_seed"], "n_sims": ST["n_sims"],
            "sit_out": W9["sit"], "trades_applied": W9["trades"], **R.code_fingerprint(Path("..").resolve())}
    R.save_entry(LOG, BACKFILL, players, teams, meta,
                 evidence={"real_games": W9["B"]["real_games"], "results": results[results["date"] <= BACKFILL]})

Recomputed vs. notebook 28's file: default version largest gap 2.3e-12, team bonuses 8.3e-17, skater WAR version 1.8e-12 (the rule: under 1e-9)
Detroit, recomputed: 80.8 points, 10.2% playoffs (notebook 28 printed 80.8, 10.2%)
Saved the entry ../data/forecast_log/2026-11-09 (1,348 players, 32 teams)


## 5. This week's update

**The evidence through yesterday** goes into the simulator's schedule and real games, **in memory only**. The bundle files are never rewritten.

**Postponed games:** a game dated on or before yesterday that wasn't played is moved to the start date, or to its new date if the daily scores show one, so it's simulated. The check below makes sure every team has 84 games (real + left).

**Notebook 28's update with the frozen settings:**

- **The default version:** goals and assists so far, the "whole" version, plus the team bonus update.
- **A second version:** also uses skater WAR so far, from your MoneyPuck file. It's logged, never the default.

In [5]:
start = time.time()
WK = run_week(EVIDENCE, START)
GC = R.games_check(WK["B"], START)
assert (GC["total"] == SE.GAMES_PER_TEAM).all(), GC[GC["total"] != SE.GAMES_PER_TEAM]
print(f"Every team: real games + games left = {SE.GAMES_PER_TEAM} (real games so far: {GC['real'].min()} to {GC['real'].max()})")
if len(WK["B"]["postponed_moved"]):
    print("Postponed games moved into the rest of the season:\n" + WK["B"]["postponed_moved"].to_string(index=False))
print(f"Trades applied (dated on or before {EVIDENCE}): {', '.join(WK['trades'])}")
print(f"Updated {len(WK['P']):,} players and 32 team bonuses in {time.time() - start:.0f} seconds "
      f"(median effective sample size {np.median(WK['U_def']['ess']):.0f} of {ST['n_guess']:,})")

Every team: real games + games left = 84 (real games so far: 16 to 20)
Trades applied (dated on or before 2026-11-10): T1, T2, T3, T4, T5, T6
Updated 1,348 players and 32 team bonuses in 35 seconds (median effective sample size 2976 of 3,000)


## 6. The reality scenario, and the same-seed check

This is **the forecast that gets graded**:

- **Start:** today.
- **Trades:** every logged trade through yesterday.
- **Sit-outs:** your list.
- **Settings:** the frozen notebook 28 update, 10,000 seasons, the frozen seed.

**The check:** running it twice with the same seed must give the same numbers.

In [6]:
t = WK["upd"]["table"].sort_values("points", ascending=False)
print(f"From {START}: {WK['upd']['games_locked']} real games locked in, {WK['upd']['games_simulated']:,} simulated")
print(t[["team", "real_gp", "real_points", "points", "p10", "p90", "playoff_pct", "division_pct", "presidents_pct"]].round(1).to_string(index=False))
check = lambda: SE.run_scenario(WK["B"], start_date=START, trade_ids=WK["trades"], sit_out=WK["sit"], n_sims=300, seed=ST["scenario_seed"],
                                updated={"playerId": WK["P"]["playerId"].to_numpy(), "z": WK["Z_def"], "bonus": WK["bonus"].to_dict()})
a_, b_ = check(), check()
assert a_["table"].equals(b_["table"]) and np.array_equal(a_["points"], b_["points"])
print("\nSame seed, same results: yes")

From 2026-11-11: 292 real games locked in, 1,052 simulated
team  real_gp  real_points  points   p10   p90  playoff_pct  division_pct  presidents_pct
 CAR       19           30   119.6 106.0 133.0         99.0          71.4            32.1
 EDM       19           29   114.5 101.0 127.0         98.2          59.9            14.2
 COL       19           24   113.2 100.0 126.0         97.7          43.2            11.7
 MIN       20           30   111.3  98.0 125.0         95.7          33.8             8.8
 BOS       18           27   109.8  96.0 124.0         91.1          34.3             7.4
 MTL       18           26   108.2  94.0 122.0         88.9          27.4             5.1
 TBL       17           24   107.0  93.0 121.0         86.7          24.4             4.4
 NJD       19           26   106.1  93.0 120.0         87.1          13.9             3.2
 VGK       19           26   105.5  91.0 119.0         90.3          22.3             3.2
 WSH       16           20   105.2  91.0 

## 7. Save this week's entry

One folder, `data/forecast_log/<evidence date>/`:

- **`players.csv`:**
  - notebook 28's columns: the evidence, preseason and updated rates in both versions, the ESS and the team bonus;
  - rest-of-season goals, assists and points (average, 10th-90th);
  - full-season points (real + simulated).
- **`teams.csv`:** preseason and updated bonus, final points (average, 10th-90th), playoff, division and Presidents' %.
- **`metadata.json`:** dates, run time, settings, seed, sit-outs, trades, MoneyPuck dates, counts, the git commit, an uncommitted-changes flag, and a sha256 of every `src/*.py` file this run used.
- **`evidence/`:** the real player-games and results through the evidence date.

**It refuses to save** if the entry already exists or roster drift (a) is unresolved. It writes to a temporary folder first and renames it in one step, so it's never half-saved.

In [7]:
lines_def, goalies_def = SE.season_lines(WK["upd"], WK["B"])
lines_war, _ = SE.season_lines(WK["uwar"], WK["B"])
PLAYERS = R.player_entry(WK["P"], WK["U_def"], WK["U_war"], WK["runs"]["bonus0"], WK["bonus"], EVIDENCE, ST, lines_def, lines_war)
TEAMS = R.team_entry(WK["upd"], WK["uwar"], WK["runs"]["bonus0"], WK["bonus"])
META = {"evidence_through": EVIDENCE, "start": START, "backfilled": False, "created": TODAY, "run_started": RUN_STARTED,
        "run_finished": datetime.now().isoformat(timespec="seconds"), "settings": ST, "seed": ST["scenario_seed"], "n_sims": ST["n_sims"],
        "sit_out": SIT.to_dict(orient="records"), "drift_acknowledged": ACKED[["playerId", "name"]].to_dict(orient="records"),
        "trades_applied": WK["trades"], "moneypuck_files": MP.to_dict(orient="records"),
        "counts": {"games_final": len(final_ids), "boxscores": int(box["game_id"].nunique()), "postponed": WK["B"]["postponed_moved"]["game_id"].tolist(),
                   "goal_mismatches": len(GOALS), "games_locked": WK["upd"]["games_locked"], "games_simulated": WK["upd"]["games_simulated"]},
        **R.code_fingerprint(Path("..").resolve())}
if ALREADY_SAVED:
    SAVED = False
    print(f"Not saved: the entry for {EVIDENCE} already exists (one entry per evidence date, never overwritten).")
elif len(DRIFT):
    SAVED = False
    print(f"Not saved: {len(DRIFT)} roster change(s) need a decision first (section 3, list 4a). "
          f"Add each to trades_2026-27.csv or to {ACK_PATH.name}, then run again.")
else:
    SAVED = R.save_entry(LOG, EVIDENCE, PLAYERS, TEAMS, META,
                         evidence={"real_games": WK["B"]["real_games"], "results": results[results["date"] <= EVIDENCE]})

Saved the entry ../data/forecast_log/2026-11-10 (1,348 players, 32 teams)


## 8. What changed since the last entry

Compared with the newest earlier entry:

- **The biggest player movers** in updated points per 60;
- **the biggest team movers** in final points and playoff %;
- **Detroit's line.**

In [8]:
earlier = [d for d in R.entries(LOG) if d < EVIDENCE]
PREV = earlier[-1] if earlier else None
if PREV is None:
    print("No earlier entry to compare with.")
else:
    p0, t0, m0 = R.load_entry(LOG, PREV)
    pm = PLAYERS.merge(p0[["playerId", "upd_g60", "upd_a60"]], on="playerId", suffixes=("", "_prev"))
    pm["P/60 now"], pm["P/60 then"] = pm["upd_g60"] + pm["upd_a60"], pm["upd_g60_prev"] + pm["upd_a60_prev"]
    pm["change"] = pm["P/60 now"] - pm["P/60 then"]
    cols = ["name", "team_now", "gp", "G", "A", "P/60 then", "P/60 now", "change"]
    print(f"Since {PREV}: biggest risers in updated points per 60")
    print(pm.sort_values("change", ascending=False).head(8)[cols].round(3).to_string(index=False))
    print("\nBiggest fallers")
    print(pm.sort_values("change").head(8)[cols].round(3).to_string(index=False))
    tm = TEAMS.merge(t0, on="team", suffixes=("", "_prev"))
    have_pts = "points_prev" in tm
    if have_pts:
        tm["points change"], tm["playoff change"] = tm["points"] - tm["points_prev"], tm["playoff_pct"] - tm["playoff_pct_prev"]
        print("\nBiggest team movers (final points):")
        show = pd.concat([tm.sort_values("points change", ascending=False).head(5), tm.sort_values("points change").head(5)])
        print(show[["team", "points_prev", "points", "points change", "playoff_pct_prev", "playoff_pct", "playoff change"]].round(1).to_string(index=False))
        d0, d1 = tm.set_index("team").loc["DET"], TEAMS.set_index("team").loc["DET"]
        print(f"\nDetroit: {d0['points_prev']:.1f} points ({d0['points_p10_prev']:.0f}-{d0['points_p90_prev']:.0f}), {d0['playoff_pct_prev']:.1f}% playoffs "
              f"on {PREV} -> {d1['points']:.1f} ({d1['points_p10']:.0f}-{d1['points_p90']:.0f}), {d1['playoff_pct']:.1f}% now")
    else:
        print("\n(The earlier entry has no team points to compare.)")

Since 2026-11-09: biggest risers in updated points per 60
             name team_now   gp   G    A  P/60 then  P/60 now  change
      Devon Toews      COL 19.0 6.0 12.0      1.626     1.729   0.103
      Cole Hutson      WSH 16.0 5.0  6.0      1.615     1.697   0.082
    Dawson Mercer      WPG 19.0 3.0  7.0      1.684     1.760   0.076
     Easton Cowan      TOR 20.0 7.0  9.0      2.198     2.272   0.073
   Nils Hoglander      NSH 18.0 3.0  4.0      1.835     1.908   0.073
    Anton Lundell      FLA 18.0 4.0  4.0      1.882     1.954   0.072
Gabriel Landeskog      COL 19.0 7.0  5.0      2.074     2.141   0.067
      Andrew Copp      DET 19.0 3.0  5.0      1.526     1.591   0.065

Biggest fallers
            name team_now   gp    G    A  P/60 then  P/60 now  change
Nathan MacKinnon      COL 19.0 10.0 20.0      4.294     4.176  -0.117
     Tim Stützle      OTT 19.0  9.0 14.0      3.123     3.052  -0.071
    Kent Johnson      CBJ 16.0  8.0  8.0      2.582     2.517  -0.065
William Nylande

## 9. Takeaways

These sentences are **written by the code from this run's numbers**. A takeaway written by hand would be wrong by next week. Read them top to bottom:

- what's new;
- what moved;
- Detroit;
- any warnings;
- whether the entry was saved.

In [9]:
say = []
new_games = len(final_ids) - (int((results["final"] & (results["date"] <= PREV)).sum()) if PREV else 0)
say.append(f"This run used every game through {EVIDENCE}: {len(final_ids)} games in all, {new_games} more than the previous entry.")
if PREV:
    up, down = pm.sort_values("change").iloc[-1], pm.sort_values("change").iloc[0]
    say.append(f"Biggest scoring riser since {PREV}: {up['name']} ({up['P/60 then']:.2f} -> {up['P/60 now']:.2f} points per 60). "
               f"Biggest faller: {down['name']} ({down['P/60 then']:.2f} -> {down['P/60 now']:.2f}).")
    if have_pts:
        tu, td = tm.sort_values("points change").iloc[-1], tm.sort_values("points change").iloc[0]
        say.append(f"Teams: {tu['team']} rose most ({tu['points change']:+.1f} projected points), {td['team']} fell most ({td['points change']:+.1f}).")
d = TEAMS.set_index("team").loc["DET"]
say.append(f"Detroit now projects to {d['points']:.1f} points (10th to 90th: {d['points_p10']:.0f} to {d['points_p90']:.0f}) "
           f"with a {d['playoff_pct']:.1f}% playoff chance, with {', '.join(SIT['name'])} sitting.")
if MP["stale"].any():
    say.append("Warning: the MoneyPuck files are more than a week older than the newest game; refresh them by hand.")
if len(GOALS):
    say.append(f"Check: {len(GOALS)} team-game(s) where the players' goals in the NHL boxscore don't add up to the final score "
               "(listed in section 3). That's a gap in the NHL's own player stats (a goal no player is credited with), "
               "not a problem with the results; it doesn't stop the run.")
if len(CALLUPS):
    say.append(f"{len(CALLUPS)} player(s) played while on no NHL roster in rosters_2026-27.csv (call-ups); that's a warning only.")
if len(DRIFT):
    say.append(f"Not saved: {len(DRIFT)} roster change(s) need logging or acknowledging first.")
say.append(f"Entry {EVIDENCE}: " + ("SAVED to data/forecast_log/" + EVIDENCE + "/." if SAVED else "not saved (see section 7)."))
print("\n".join(f"- {s}" for s in say))

- This run used every game through 2026-11-10: 292 games in all, 7 more than the previous entry.
- Biggest scoring riser since 2026-11-09: Devon Toews (1.63 -> 1.73 points per 60). Biggest faller: Nathan MacKinnon (4.29 -> 4.18).
- Teams: FLA rose most (+2.7 projected points), LAK fell most (-3.1).
- Detroit now projects to 78.8 points (10th to 90th: 65 to 93) with a 7.1% playoff chance, with Dylan Larkin, Simon Edvinsson sitting.
- Check: 1 team-game(s) where the players' goals in the NHL boxscore don't add up to the final score (listed in section 3). That's a gap in the NHL's own player stats (a goal no player is credited with), not a problem with the results; it doesn't stop the run.
- 4 player(s) played while on no NHL roster in rosters_2026-27.csv (call-ups); that's a warning only.
- Entry 2026-11-10: SAVED to data/forecast_log/2026-11-10/.
